# MTP evaluation on Kaggle

Runs `scripts/evaluate.py` on published runs and shows the INTERFACES §10 JSON.

**Settings (right sidebar):** Accelerator **GPU T4 x2** (one is enough), Internet **On**.
**Secrets** (Add-ons → Secrets): `HF_TOKEN`, from an HF account that has accepted the terms of
https://huggingface.co/datasets/facebook/flores. Without it, only the IndicCorp datasets work.
**Inputs:** *Add Input →* the run datasets to evaluate (e.g. `jainamrampuria/mtp-run-R2`). Every attached
folder with a `config.yaml` is treated as a run.

Afterwards, download `results/` from the Output tab, copy the `eval_*.json` files into the repo's
`results/` and commit them. Token dumps (`tokens_*.jsonl`) are large; keep them out of git.

In [ ]:
# 1. parameters
DATASETS = ["indiccorp_eval", "flores_hi"]   # also: indiccorp_eval_small, flores_mr
GROUPER = ""                                  # empty = data.grouper from each run's config
STEP = ""                                     # empty = latest step
DUMP_TOKENS = False                           # per-token dump for Jai's error analysis
BRANCH = "main"

In [ ]:
# 2. secrets, clone, install
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("no HF_TOKEN secret: FLORES will fail (gated); IndicCorp still works")

%cd /kaggle/working
!rm -rf /tmp/mtp && git clone -q --depth 1 --branch {BRANCH} https://github.com/rampuriajainam/mtp-indic-wordgroup.git /tmp/mtp
%cd /tmp/mtp
!git log -1 --oneline
!pip uninstall -y -q torchao   # Kaggle ships torchao 0.10; peft 0.21 refuses LoRA when it is present
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. find attached runs
import glob
RUNS = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.yaml", recursive=True)})
print("\n".join(RUNS) or "no run attached: Add Input -> mtp-run-<ID>")

In [ ]:
# 4. evaluate each run (results go to /kaggle/working/results so they are in the Output tab)
ds = " ".join(DATASETS)
for run in RUNS:
    extra = (f"--grouper {GROUPER} " if GROUPER else "") + (f"--step {STEP} " if STEP else "") + ("--dump_tokens" if DUMP_TOKENS else "")
    !PYTHONUTF8=1 python scripts/evaluate.py --run_dir {run} --datasets {ds} --out /kaggle/working/results {extra}

In [ ]:
# 5. show the JSON
import json
for path in sorted(glob.glob("/kaggle/working/results/*/eval_*.json")):
    rec = json.load(open(path, encoding="utf-8"))
    print(f"\n{rec['run_name']} | {rec['dataset']} | step {rec['step']} | grouper {rec['grouper']}")
    for r in rec["per_head"]:
        ig = r["top1_in_group"]; bd = r["top1_at_boundary"]
        print(f"  head {r['head']}: loss {r['loss']:.3f}  top1 {100*r['top1']:.1f}%  top5 {100*r['top5']:.1f}%"
              + (f"  in-group {100*ig:.1f}% / boundary {100*bd:.1f}%" if ig is not None and bd is not None else ""))